# Строгий датасет: Сфера → адрес → CDI → ЕГРН

## Что делает ноутбук

Ноутбук собирает строгий датасет объектов недвижимости и дополняет его сведениями ЕГРН. Одна строка итоговой таблицы — один объект Сферы в одном договоре.

Полный путь данных:

```text
Сфера: договор → заявка → задача оформления → объект недвижимости
CDI: адрес объекта Сферы → ФИАС дома
ЕГРН: ФИАС дома → здание → сведения ЕГРН
```

CDI ID страхователя в соединении не используется. В CDI передаётся только `full_address` объекта. Если `full_address` не заполнен, строка остаётся в датасете без связи с CDI и ЕГРН.

## Как принимается решение

1. По тексту адреса CDI ищет адресные записи.
2. Из ответа берётся ФИАС дома.
3. Если CDI вернул один ФИАС дома, поиск продолжается в ЕГРН.
4. Если ФИАС дома нет или найдено несколько разных ФИАС, ЕГРН не присоединяется.
5. В ЕГРН по ФИАС дома ищутся только здания, строения и сооружения. Помещения, квартиры и офисы не используются.
6. Если найдено одно здание ЕГРН, оно присоединяется без проверки площади.
7. Если зданий несколько и площадь Сферы заполнена, кандидаты сравниваются по площади.
8. Если после проверки площади осталось одно здание, оно присоединяется.
9. Если кандидатов всё равно несколько, поля ЕГРН остаются пустыми.

Если площади в Сфере нет, это не мешает соединению при одном кандидате ЕГРН. Площадь нужна только для уточнения, когда по ФИАС дома найдено несколько зданий.

Объект Сферы не удаляется, даже если CDI или ЕГРН ничего не нашли. Количество строк до и после соединения должно совпадать.

КХД используется только на чтение. Временные таблицы не создаются.


In [1]:
%pip install pandas sqlalchemy "psycopg[binary]" oracledb

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import re
from pathlib import Path
import oracledb
import pandas as pd
from sqlalchemy import URL, create_engine, text

pd.set_option('display.max_columns', 100)

In [3]:
CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / 'уч данные.txt').exists():
    NOTEBOOK_DIR = CURRENT_DIR
elif (CURRENT_DIR / 'notebooks' / 'уч данные.txt').exists():
    NOTEBOOK_DIR = CURRENT_DIR / 'notebooks'
else:
    NOTEBOOK_DIR = CURRENT_DIR

PROJECT_ROOT = (
    NOTEBOOK_DIR.parent
    if NOTEBOOK_DIR.name == 'notebooks'
    else NOTEBOOK_DIR
)
OUTPUT_DIR = PROJECT_ROOT / 'РЕЗУЛЬТАТЫ_ЛОКАЛЬНО'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Корень проекта:', PROJECT_ROOT)
print('Папка результатов:', OUTPUT_DIR)

Корень проекта: t:\Блок актуарных расчетов\Управление актуарных расчетов\Общая\Светова\риск моделирование\анализ таблиц\sql python
Папка результатов: t:\Блок актуарных расчетов\Управление актуарных расчетов\Общая\Светова\риск моделирование\анализ таблиц\sql python\РЕЗУЛЬТАТЫ_ЛОКАЛЬНО


# 1. Подключение к Сфере


In [4]:
CREDENTIALS_PATH = NOTEBOOK_DIR / 'уч данные.txt'

def read_credentials(path):
    if not path.exists():
        raise FileNotFoundError(f'Не найден файл с учётными данными: {path}')

    credentials = {}
    for line_number, raw_line in enumerate(
        path.read_text(encoding='utf-8-sig').splitlines(),
        start=1,
    ):
        line = raw_line.strip()
        if not line or line.startswith('#'):
            continue
        if '=' not in line:
            raise ValueError(
                f'Строка {line_number}: ожидается запись КЛЮЧ=значение'
            )

        key, value = line.split('=', 1)
        credentials[key.strip()] = value.strip()

    return credentials

credentials = read_credentials(CREDENTIALS_PATH)

sphere_required = [
    'SPHERE_HOST',
    'SPHERE_DATABASE',
    'SPHERE_USER',
    'SPHERE_PASSWORD',
]
sphere_missing = [key for key in sphere_required if not credentials.get(key)]
if sphere_missing:
    raise ValueError(
        'Заполни в уч данные.txt: ' + ', '.join(sphere_missing)
    )

SPHERE_HOST = credentials['SPHERE_HOST']
SPHERE_PORT = int(credentials.get('SPHERE_PORT', '5432'))
SPHERE_DATABASE = credentials['SPHERE_DATABASE']
SPHERE_USER = credentials['SPHERE_USER']
SPHERE_PASSWORD = credentials['SPHERE_PASSWORD']

connection_url = URL.create(
    drivername='postgresql+psycopg',
    username=SPHERE_USER,
    password=SPHERE_PASSWORD,
    host=SPHERE_HOST,
    port=SPHERE_PORT,
    database=SPHERE_DATABASE,
)
engine = create_engine(connection_url, pool_pre_ping=True)

print('Учётные данные прочитаны, подключение к Сфере создано')


Учётные данные прочитаны, подключение к Сфере создано


In [5]:
with engine.connect() as connection:
    connection_check = pd.read_sql_query(
        text('select current_database() as database_name, current_user as user_name'),
        connection,
    )

display(connection_check)

,database_name,user_name
0,postgres,svetovavs


# 2. Подключение к Oracle КХД



In [6]:
khd_required = [
    'KHD_HOST',
    'KHD_SERVICE_NAME',
    'KHD_USER',
    'KHD_PASSWORD',
]
khd_missing = [key for key in khd_required if not credentials.get(key)]
if khd_missing:
    raise ValueError(
        'Заполни в уч данные.txt: ' + ', '.join(khd_missing)
    )

KHD_HOST = credentials['KHD_HOST']
KHD_PORT = int(credentials.get('KHD_PORT', '1521'))
KHD_SERVICE_NAME = credentials['KHD_SERVICE_NAME']
KHD_USER = credentials['KHD_USER']
KHD_PASSWORD = credentials['KHD_PASSWORD']
KHD_DATA_SCHEMA = credentials.get('KHD_DATA_SCHEMA', 'DM_RISK_AVATAR')

khd_dsn = oracledb.makedsn(
    KHD_HOST,
    KHD_PORT,
    service_name=KHD_SERVICE_NAME,
)
khd_connection = oracledb.connect(
    user=KHD_USER,
    password=KHD_PASSWORD,
    dsn=khd_dsn,
)

print('Подключение к КХД создано')


Подключение к КХД создано


In [7]:
# проверяем доступ к таблицам CDI и ЕГРН
khd_schema_for_check = KHD_DATA_SCHEMA.upper()
if not khd_schema_for_check.replace('_', '').isalnum():
    raise ValueError('Некорректное имя схемы КХД')

tables_for_check = [
    'STG_ADDRESS_CDI_ZUD',
    'STG_PARTY_SRC',
    'EGRN_DATA',
]
with khd_connection.cursor() as cursor:
    for table_name in tables_for_check:
        cursor.execute(
            f'select 1 from {khd_schema_for_check}.{table_name} '
            'where rownum = 1'
        )
        cursor.fetchone()
        print(f'Таблица {table_name} доступна')


Таблица STG_ADDRESS_CDI_ZUD доступна
Таблица STG_PARTY_SRC доступна
Таблица EGRN_DATA доступна


# 3. SQL Сфера, строгий подход

In [8]:
strict_sql = r"""
/*
Для чего нужен запрос
---------------------
Запрос собирает основу датасета для модели 1 по недвижимости ЮЛ.
Он объединяет сведения о договоре, объекте, адресе, страхователе,
отрасли, страховых суммах и ближайшем предыдущем договоре.

Одна строка результата
----------------------
Одна строка - один объект недвижимости в одном договоре.
Один договор может занимать несколько строк, если в нем несколько объектов.


Как связаны таблицы
-------------------
Договор -> заявка -> задача оформления -> объект в задаче
        -> характеристики объекта -> сам объект -> адрес
        -> условия страхования объекта

Из договора берется страхователь. Из заявки берется CRM-карточка,
в которой находятся отрасль и сегмент.

Какие записи попадают в результат
---------------------------------
- задача оформления договора: task_type = draft_contract;
- завершенная рабочая задача: status = operational_archive;
- тип документа: ins_document_type = new_ins_contract,
  ins_contract_prolong или NULL, то есть новый договор, пролонгация
  или незаполненное значение;
- отказ в страховании не установлен: ins_refuse IS NOT TRUE;
- дата удаления отсутствует: d_delete IS NULL для задачи, заявки,
  договора и объекта;
- тип объекта: elementary_obj_type = nedv_ul_and_ip,
  то есть недвижимость ЮЛ и ИП.


Как читать страховые суммы
--------------------------
- contract_insured_sum - общая СС всего договора;
- task_object_insured_sum - СС объекта в строке связи задачи и объекта;
- condition_min_insured_sum и condition_max_insured_sum - минимальная и
  максимальная СС среди условий выбранной версии объекта;
- insured_sum - СС среди условий выбранной версии объекта.


Как используется история
-------------------------
Ближайший предыдущий договор ищется по bps_contract.prevcontract_id.
Прошлая СС объекта заполняется только тогда, когда в текущем и предыдущем
договоре совпал object_id. Если при пролонгации объект завели с новым ID,
прошлая объектная СС останется пустой.

*/

with task_candidates as (
    /* Шаг 1. Находим все подходящие задачи оформления. */
    select
        c.id as contract_id,
        c.n_contract as contract_number,
        c.prevcontract_id as previous_contract_id,
        c.rootcontract_id as root_contract_id,
        c.contractor_id as policyholder_id,
        c.document_status as contract_status,
        c.d_sign_contract as contract_sign_date,
        c.d_start_contract as contract_start_date,
        c.d_end_contract as contract_end_date,
        c.currency as contract_currency,
        c.ins_product_sbs as insurance_product,
        c.ins_program as insurance_program,

        r.id as request_id,
        r.corporate_crm_id,
        r.business_segment,

        t.id as task_id,
        t.d_create as task_create_date,
        t.d_change as task_change_date,
        t.task_type,
        t.status as task_status,
        t.ins_document_type,
        t.contract_type,
        t.d_conclusion_ins_contract as contract_conclusion_date,
        t.ins_refuse,
        t.industry as task_industry,
        t.subindustry as task_subindustry,
        t.total_ins_contract_amount as contract_insured_sum,
        t.total_ins_contract_premium as contract_premium,
        t.curr_ins_contract_amount as contract_amount_currency,

        coalesce(
            t.d_conclusion_ins_contract::timestamp with time zone,
            c.d_sign_contract,
            t.d_create
        ) as as_of_date,

        row_number() over (
            partition by c.id
            order by
                coalesce(
                    t.d_conclusion_ins_contract::timestamp with time zone,
                    t.d_create,
                    t.d_change
                ) desc nulls last,
                t.d_create desc nulls last,
                t.d_change desc nulls last,
                t.id desc
        ) as task_number
    from bps_request_ins_task t
    join bps_request_ins r
        on r.id = t.request_ins_id
    join bps_contract c
        on c.id = r.contract_id
    where t.task_type = 'draft_contract'
      and t.status = 'operational_archive'
      and (
          t.ins_document_type = 'new_ins_contract'
          or t.ins_document_type = 'ins_contract_prolong'
          or t.ins_document_type is null
      )
      and t.ins_refuse is not true
      and t.d_delete is null
      and r.d_delete is null
      and c.d_delete is null
),

selected_tasks as (
    /* Шаг 2. Для каждого договора оставляем одну самую позднюю задачу. */
    select *
    from task_candidates
    where task_number = 1
),

contract_context as (
    /* Шаг 3. Добавляем ближайший предыдущий договор, если он указан. */
    select
        current_task.*,
        previous_contract.n_contract as previous_contract_number,
        previous_contract.d_start_contract as previous_contract_start_date,
        previous_contract.d_end_contract as previous_contract_end_date,
        previous_task.contract_insured_sum as previous_contract_insured_sum,
        previous_task.contract_premium as previous_contract_premium,
        previous_task.contract_amount_currency
            as previous_contract_amount_currency
    from selected_tasks current_task
    left join bps_contract previous_contract
        on previous_contract.id = current_task.previous_contract_id
    left join selected_tasks previous_task
        on previous_task.contract_id = current_task.previous_contract_id
),

object_candidates as (
    /*
    Шаг 4. К выбранной задаче присоединяем объекты недвижимости.
    Здесь же добавляем адрес, страхователя, CRM и характеристики объекта.
    */
    select
        contract.*,

        policyholder.inn as policyholder_inn,
        policyholder.contractor_type as policyholder_type,
        policyholder.cdi_id as policyholder_cdi_id,
        policyholder.ogrn as policyholder_ogrn,
        policyholder.kpp as policyholder_kpp,
        policyholder.company_name_short as policyholder_name,
        policyholder.company_form as policyholder_company_form,
        policyholder.company_register_day
            as policyholder_registration_date,

        crm.id as crm_id,
        crm.client_id as crm_client_id,
        crm.segment as crm_segment,
        crm.macroindustry as crm_macroindustry,
        crm.industry as crm_industry,
        crm.primary_occupation as crm_primary_occupation,
        crm.specialization as crm_specialization,
        crm.okved as crm_okved,

        link.id as task_object_link_id,
        link.characteristics_id,
        link.object_group_id,
        link.insured_sum as task_object_insured_sum,
        link.insured_sum_currency as task_object_insured_sum_currency,
        link.per_occurance_limit as task_object_per_occurrence_limit,

        obj.id as object_id,
        obj.obj_name as object_name,
        obj.description as object_description,
        obj.obj_type as object_type,
        obj.elementary_obj_type,
        obj.original_address,
        obj.geo_address_id,

        ch.version_number as characteristics_version_number,
        ch.version_start_date as characteristics_version_start_date,
        ch.version_end_date as characteristics_version_end_date,
        ch.version_is_active as characteristics_version_is_active,
        ch.insurance_value,
        ch.insurance_value_currency,
        ch.insurance_value_basis,
        ch.is_pledged,
        ch.pledged_value,
        ch.ownership_type,
        ch.is_leased,
        ch.insured_components,
        ch.activity_types,
        ch.risk_natures,
        ch.insurance_territory,
        ch.characteristics ->> 'total_area_sq_m' as total_area,
        ch.characteristics ->> 'occupied_area_sq_m' as occupied_area,
        ch.characteristics ->> 'construction_year' as construction_year,
        ch.characteristics ->> 'last_capital_repair_year'
            as capital_repair_year,
        ch.characteristics ->> 'total_floors_count' as floors_count,
        ch.characteristics ->> 'occupied_floor' as occupied_floor,
        ch.characteristics ->> 'load_bearing_walls_material'
            as walls_material,
        ch.characteristics ->> 'interfloor_overlap_material'
            as overlap_material,
        ch.characteristics ->> 'roofing_material' as roofing_material,
        ch.characteristics as object_characteristics_json,

        address.full_address,
        address.postal_code,
        address.region_id as address_region_id,
        address.area as district,
        address.settlement,
        address.street,
        address.house,
        address.building,
        address.block,
        address.flat,
        address.office,
        address.fias_code,
        address.longitude,
        address.latitude,
        address.address_dgis_id,

        row_number() over (
            partition by contract.task_id, obj.id
            order by
                link.d_change desc nulls last,
                link.d_create desc nulls last,
                ch.version_start_date desc nulls last,
                ch.version_number desc nulls last,
                link.id desc,
                ch.id desc
        ) as object_number
    from contract_context contract
    join bps_request_ins_task_insurance_object link
        on link.parent_id = contract.task_id
    join base_insurance_object_characteristics ch
        on ch.id = link.characteristics_id
    join base_insurance_object obj
        on obj.id = ch.insurance_object_id
    left join base_geo_address address
        on address.id = obj.geo_address_id
    left join bps_contractor policyholder
        on policyholder.id = contract.policyholder_id
    left join bps_corporate_crm crm
        on crm.id = contract.corporate_crm_id
    where obj.elementary_obj_type = 'nedv_ul_and_ip'
      and obj.d_delete is null
),

selected_objects as (
    /*
    Шаг 5. Если объект несколько раз связан с одной задачей,
    оставляем одну самую позднюю запись связи.
    */
    select *
    from object_candidates
    where object_number = 1
),

selected_characteristics as (
    /* Шаг 6. Получаем список версий объектов для поиска их условий. */
    select distinct characteristics_id
    from selected_objects
),

condition_summary as (
    /*
    Шаг 7. У одной версии объекта может быть несколько вариантов условий.
    Сворачиваем их в одну строку, чтобы один объект не продублировался.
    */
    select
        cond.characteristics_id,
        count(*) as condition_count,
        min(cond.insured_sum) as condition_min_insured_sum,
        max(cond.insured_sum) as condition_max_insured_sum,
        count(distinct cond.insured_sum_currency) filter (
            where cond.insured_sum_currency is not null
        ) as condition_currency_count,
        string_agg(
            distinct cond.insured_sum_currency,
            ', '
            order by cond.insured_sum_currency
        ) filter (
            where cond.insured_sum_currency is not null
        ) as insured_sum_currency,
        min(cond.per_occurance_limit) as minimum_per_occurrence_limit,
        max(cond.per_occurance_limit) as maximum_per_occurrence_limit,
        jsonb_agg(
            jsonb_strip_nulls(
                jsonb_build_object(
                    'option_number', cond.terms_option_number,
                    'insured_sum', cond.insured_sum,
                    'currency', cond.insured_sum_currency,
                    'per_occurrence_limit', cond.per_occurance_limit
                )
            )
            order by cond.terms_option_number nulls last, cond.id
        ) as conditions_json
    from base_insurance_object_conditions cond
    join selected_characteristics selected
        on selected.characteristics_id = cond.characteristics_id
    group by cond.characteristics_id
),

object_data as (
    /* Шаг 8. Добавляем к каждому объекту найденные суммы и условия. */
    select
        obj.*,
        conditions.condition_count,
        conditions.condition_min_insured_sum,
        conditions.condition_max_insured_sum,
        conditions.condition_currency_count,
        conditions.insured_sum_currency,
        conditions.minimum_per_occurrence_limit,
        conditions.maximum_per_occurrence_limit,
        conditions.conditions_json
    from selected_objects obj
    left join condition_summary conditions
        on conditions.characteristics_id = obj.characteristics_id
),

objects_with_previous as (
    /*
    Шаг 9. Ищем тот же object_id в ближайшем предыдущем договоре
    и, если нашли, добавляем его предыдущую СС.
    */
    select
        current_object.*,
        case
            when previous_object.condition_min_insured_sum =
                 previous_object.condition_max_insured_sum
             and previous_object.condition_currency_count <= 1
            then previous_object.condition_max_insured_sum
        end as previous_object_insured_sum,
        previous_object.insured_sum_currency
            as previous_object_insured_sum_currency
    from object_data current_object
    left join object_data previous_object
        on previous_object.contract_id = current_object.previous_contract_id
       and previous_object.object_id = current_object.object_id
),

raw_result as (
/* Шаг 10. Собираем исходные поля строгого датасета. */
select
    /* Основные ID. */
    obj.contract_id,
    obj.contract_number,
    obj.previous_contract_id,
    obj.root_contract_id,
    obj.request_id,
    obj.task_id,
    obj.task_object_link_id,
    obj.characteristics_id,
    obj.object_id,
    obj.geo_address_id,
    obj.policyholder_id,
    obj.corporate_crm_id,

    /* Договор и его даты. */
    obj.as_of_date,
    obj.contract_conclusion_date,
    obj.contract_sign_date,
    obj.contract_start_date,
    obj.contract_end_date,
    obj.contract_status,
    obj.ins_document_type,
    obj.contract_type,
    obj.contract_currency,
    obj.insurance_product,
    obj.insurance_program,

    /* Объект. */
    count(*) over (
        partition by obj.contract_id
    ) as real_estate_objects_in_contract,
    obj.object_group_id,
    obj.object_name,
    obj.object_description,
    obj.object_type,
    obj.elementary_obj_type,
    obj.total_area,
    obj.occupied_area,
    obj.construction_year,
    obj.capital_repair_year,
    obj.floors_count,
    obj.occupied_floor,
    obj.walls_material,
    obj.overlap_material,
    obj.roofing_material,
    obj.ownership_type,
    obj.is_leased,
    obj.insured_components,
    obj.activity_types,
    obj.risk_natures,
    obj.insurance_territory,

    /* Адрес. */
    obj.full_address,
    obj.original_address,
    obj.postal_code,
    obj.address_region_id,
    obj.district,
    obj.settlement,
    obj.street,
    obj.house,
    obj.building,
    obj.block,
    obj.flat,
    obj.office,
    obj.fias_code,
    obj.longitude,
    obj.latitude,
    obj.address_dgis_id,

    /* Страхователь, отрасль и сегмент. */
    obj.policyholder_inn,
    obj.policyholder_type,
    obj.policyholder_cdi_id,
    obj.policyholder_ogrn,
    obj.policyholder_kpp,
    obj.policyholder_name,
    obj.policyholder_company_form,
    obj.policyholder_registration_date,
    obj.crm_id,
    obj.crm_client_id,
    (obj.crm_client_id = obj.policyholder_id) as crm_client_is_policyholder,
    obj.crm_segment,
    obj.crm_macroindustry,
    obj.crm_industry,
    obj.crm_primary_occupation,
    obj.crm_specialization,
    obj.crm_okved,
    obj.business_segment,
    obj.task_industry,
    obj.task_subindustry,

    /*
    Все СС стоят рядом.
    СС договора относится ко всему договору и повторяется у его объектов.
    */
    obj.contract_insured_sum,
    obj.contract_amount_currency,
    min(obj.condition_min_insured_sum) over (
        partition by obj.contract_id
    ) as contract_real_estate_min_insured_sum,
    max(obj.condition_max_insured_sum) over (
        partition by obj.contract_id
    ) as contract_real_estate_max_insured_sum,
    obj.task_object_insured_sum,
    obj.task_object_insured_sum_currency,
    obj.condition_min_insured_sum,
    obj.condition_max_insured_sum,
    case
        /* Не выбираем случайную СС, если в условиях есть расхождения. */
        when obj.condition_min_insured_sum =
             obj.condition_max_insured_sum
         and obj.condition_currency_count <= 1
        then obj.condition_max_insured_sum
    end as insured_sum,
    obj.insured_sum_currency,
    obj.condition_currency_count,
    obj.previous_contract_insured_sum,
    obj.previous_contract_amount_currency,
    obj.previous_object_insured_sum,
    obj.previous_object_insured_sum_currency,

    /* Премии, стоимости и лимиты. */
    obj.contract_premium,
    obj.previous_contract_premium,
    obj.insurance_value,
    obj.insurance_value_currency,
    obj.insurance_value_basis,
    obj.is_pledged,
    obj.pledged_value,
    obj.task_object_per_occurrence_limit,
    obj.minimum_per_occurrence_limit,
    obj.maximum_per_occurrence_limit,

    /* Простая договорная история. */
    (obj.previous_contract_id is not null) as has_previous_contract,
    obj.previous_contract_number,
    obj.previous_contract_start_date,
    obj.previous_contract_end_date,

    /* Исходные данные для проверки. */
    obj.condition_count,
    obj.conditions_json,
    obj.characteristics_version_number,
    obj.characteristics_version_start_date,
    obj.characteristics_version_end_date,
    obj.characteristics_version_is_active,
    obj.object_characteristics_json,
    obj.task_type,
    obj.task_status,
    obj.ins_refuse
from objects_with_previous obj
),

standardized_result as (
    /* Шаг 11. Приводим результат к общей структуре двух датасетов. */
    select
        case
            when raw.contract_id is null then 'not_linked'
            else 'linked'
        end as row_source,
        case
            when count(raw.contract_id) over (
                partition by raw.object_id
            ) = 0 then 'not_linked'
            when count(raw.contract_id) over (
                partition by raw.object_id
            ) = 1 then 'linked'
            else 'multiple_contracts'
        end as contract_link_status,
        count(raw.contract_id) over (
            partition by raw.object_id
        ) as contract_count,
        (raw.contract_id is not null) as has_contract,
        (
            raw.geo_address_id is not null
            or nullif(btrim(raw.full_address), '') is not null
            or nullif(btrim(raw.original_address), '') is not null
        ) as has_address,
        (raw.insured_sum is not null) as has_target,
        case
            when raw.condition_count is null
              or raw.condition_count = 0
                then 'no_conditions'
            when raw.condition_min_insured_sum is distinct from
                 raw.condition_max_insured_sum
                then 'several_target_values'
            when coalesce(raw.condition_currency_count, 0) > 1
                then 'several_currencies'
            when raw.insured_sum <= 0
                then 'target_is_not_positive'
            when raw.insured_sum is null
                then 'target_is_empty'
            else 'target_is_usable'
        end as target_status,

        raw.contract_id,
        raw.contract_number,
        raw.previous_contract_id,
        raw.root_contract_id,
        raw.request_id,
        raw.task_id,
        raw.task_object_link_id,
        raw.characteristics_id,
        raw.object_id,
        raw.geo_address_id,
        raw.policyholder_id,
        raw.corporate_crm_id,

        raw.as_of_date,
        raw.contract_conclusion_date,
        raw.contract_sign_date,
        raw.contract_start_date,
        raw.contract_end_date,
        raw.contract_status,
        raw.ins_document_type,
        raw.insurance_product,

        case
            when raw.contract_id is not null then
                count(raw.object_id) over (
                    partition by raw.contract_id
                )
        end as real_estate_objects_in_contract,
        raw.object_name,
        raw.object_description,
        raw.object_type,
        raw.elementary_obj_type,
        raw.total_area,
        raw.occupied_area,
        raw.construction_year,
        raw.capital_repair_year,
        raw.floors_count,
        raw.occupied_floor,
        raw.walls_material,
        raw.overlap_material,
        raw.roofing_material,
        raw.ownership_type,
        raw.is_leased,
        raw.insured_components,
        raw.activity_types,
        raw.risk_natures,
        raw.insurance_territory,

        raw.full_address,
        raw.original_address,
        raw.postal_code,
        raw.address_region_id,
        raw.district,
        raw.settlement,
        raw.street,
        raw.house,
        raw.building,
        raw.block,
        raw.flat,
        raw.office,
        raw.fias_code,
        raw.longitude,
        raw.latitude,
        raw.address_dgis_id,

        raw.policyholder_inn,
        raw.policyholder_name,
        raw.policyholder_cdi_id,
        raw.crm_segment,
        raw.crm_macroindustry,
        raw.crm_industry,
        raw.crm_okved,
        raw.business_segment,
        raw.task_industry,
        raw.task_subindustry,

        raw.contract_insured_sum,
        raw.contract_amount_currency,
        raw.task_object_insured_sum,
        raw.task_object_insured_sum_currency,
        raw.condition_min_insured_sum,
        raw.condition_max_insured_sum,
        raw.insured_sum,
        raw.insured_sum_currency,
        raw.condition_currency_count,
        raw.contract_premium,
        raw.insurance_value,
        raw.insurance_value_currency,
        raw.insurance_value_basis,
        raw.is_pledged,
        raw.pledged_value,
        raw.minimum_per_occurrence_limit,
        raw.maximum_per_occurrence_limit,

        raw.previous_contract_number,
        raw.previous_contract_start_date,
        raw.previous_contract_end_date,

        raw.condition_count,
        raw.characteristics_version_number,
        raw.characteristics_version_start_date,
        raw.characteristics_version_end_date,
        raw.characteristics_version_is_active,
        raw.object_characteristics_json,
        raw.task_type,
        raw.task_status,
        raw.ins_refuse
    from raw_result raw
)

select *
from standardized_result
order by
    as_of_date desc nulls last,
    contract_id,
    object_id;

"""


In [9]:
with engine.connect() as connection:
    strict_df = pd.read_sql_query(text(strict_sql), connection)

print('Строк:', len(strict_df))
print('Колонок:', len(strict_df.columns))
display(strict_df.head(3))


Строк: 1211
Колонок: 102


,row_source,contract_link_status,contract_count,has_contract,has_address,has_target,target_status,contract_id,contract_number,previous_contract_id,root_contract_id,request_id,task_id,task_object_link_id,characteristics_id,object_id,geo_address_id,policyholder_id,corporate_crm_id,as_of_date,contract_conclusion_date,contract_sign_date,contract_start_date,contract_end_date,contract_status,ins_document_type,insurance_product,real_estate_objects_in_contract,object_name,object_description,object_type,elementary_obj_type,total_area,occupied_area,construction_year,capital_repair_year,floors_count,occupied_floor,walls_material,overlap_material,roofing_material,ownership_type,is_leased,insured_components,activity_types,risk_natures,insurance_territory,full_address,original_address,postal_code,...,settlement,street,house,building,block,flat,office,fias_code,longitude,latitude,address_dgis_id,policyholder_inn,policyholder_name,policyholder_cdi_id,crm_segment,crm_macroindustry,crm_industry,crm_okved,business_segment,task_industry,task_subindustry,contract_insured_sum,contract_amount_currency,task_object_insured_sum,task_object_insured_sum_currency,condition_min_insured_sum,condition_max_insured_sum,insured_sum,insured_sum_currency,condition_currency_count,contract_premium,insurance_value,insurance_value_currency,insurance_value_basis,is_pledged,pledged_value,minimum_per_occurrence_limit,maximum_per_occurrence_limit,previous_contract_number,previous_contract_start_date,previous_contract_end_date,condition_count,characteristics_version_number,characteristics_version_start_date,characteristics_version_end_date,characteristics_version_is_active,object_characteristics_json,task_type,task_status,ins_refuse
0,linked,linked,1,True,True,True,target_is_usable,116926949,013БС4040041498,None,None,441239,697050,35323,29926,29948,15343.0,24663764,4162.0,2026-08-27 21:00:00+00:00,2026-08-28,2025-08-27 21:00:00+00:00,2025-08-28 21:00:00+00:00,2026-08-28 20:59:59.999999+00:00,Оформлен,ins_contract_prolong,СББ. Имущество юридических лиц (залоги) - свер...,3,NaN,NaN,prop_assets_legal_entities,nedv_ul_and_ip,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,False,[None],[],[],None,"заволжье, советская улица, 1а к5",NaN,NaN,...,None,NaN,NaN,None,None,None,None,None,43.422003,56.639913,70030076201844929,7327077188,"ООО ""УАЗ""",None,cib,transport_and_automotive,NaN,NaN,cib,mechanical_engineering,car,0.0,RUB,NaN,NaN,1.443507e+08,1.443507e+08,1.443507e+08,RUB,1,3682709.35,1.443507e+08,RUB,NaN,False,None,None,None,None,None,None,1,1,2026-08-14 07:19:16.011901+00:00,None,True,{},draft_contract,operational_archive,False
1,linked,linked,1,True,True,True,target_is_usable,116926949,013БС4040041498,None,None,441239,697050,35324,29928,29950,15344.0,24663764,4162.0,2026-08-27 21:00:00+00:00,2026-08-28,2025-08-27 21:00:00+00:00,2025-08-28 21:00:00+00:00,2026-08-28 20:59:59.999999+00:00,Оформлен,ins_contract_prolong,СББ. Имущество юридических лиц (залоги) - свер...,3,NaN,NaN,prop_assets_legal_entities,nedv_ul_and_ip,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,False,[None],[],[],None,"заволжье, советская улица, 1а к3",NaN,NaN,...,None,NaN,NaN,None,None,None,None,None,43.414104,56.640429,70030076201841531,7327077188,"ООО ""УАЗ""",None,cib,transport_and_automotive,NaN,NaN,cib,mechanical_engineering,car,0.0,RUB,NaN,NaN,1.725535e+08,1.725535e+08,1.725535e+08,RUB,1,3682709.35,1.725535e+08,RUB,NaN,False,None,None,None,None,None,None,1,1,2026-08-14 07:23:54.622079+00:00,None,True,{},draft_contract,operational_archive,False
2,linked,linked,1,True,True,True,target_is_usable,116926949,013БС4040041498,None,None,441239,697050,35325,29933,29955,15346.0,24663764,4162.0,2026-08-27 21:00:00+00:00,2026-08-28,2025-08-27 21:00:00+00:00,2025-08-28 21:00:00+00:00,2026-08-28 20:59:59.999999+00:00,Оформлен,ins_contract_prolong,СББ. Имущество юридических лиц (залоги) - свер...,3,NaN,NaN,prop_assets_legal_entities,nedv_ul_and_ip,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,False,[None],[],[],None,"заволжье, советск

# 4. Проверка заполненности

In [10]:
required_columns = {
    'contract_id', 'task_id', 'object_id', 'characteristics_id',
    'elementary_obj_type', 'insured_sum', 'full_address'
}
missing_columns = sorted(required_columns - set(strict_df.columns))
if missing_columns:
    raise ValueError('Не найдены ожидаемые колонки: ' + ', '.join(missing_columns))

profile = pd.DataFrame({
    'Показатель': [
        'Строк',
        'Уникальных договоров',
        'Уникальных задач',
        'Уникальных объектов',
        'Уникальных пар задача + объект',
        'Строк с target',
        'Строк с адресом',
        'Строк с пустым типом объекта',
    ],
    'Значение': [
        len(strict_df),
        strict_df['contract_id'].nunique(dropna=True),
        strict_df['task_id'].nunique(dropna=True),
        strict_df['object_id'].nunique(dropna=True),
        strict_df[['task_id', 'object_id']].drop_duplicates().shape[0],
        strict_df['insured_sum'].notna().sum(),
        strict_df['full_address'].fillna('').str.strip().ne('').sum(),
        strict_df['elementary_obj_type'].fillna('').str.strip().eq('').sum(),
    ],
})

display(profile)

,Показатель,Значение
0,Строк,1211
1,Уникальных договоров,570
2,Уникальных задач,570
3,Уникальных объектов,1201
4,Уникальных пар задача + объект,1211
5,Строк с target,1169
6,Строк с адресом,1057
7,Строк с пустым типом объекта,0


In [11]:
duplicate_keys = (
    strict_df.groupby(['task_id', 'object_id'], dropna=False)
    .size()
    .gt(1)
    .sum()
)
print('Повторных ключей задача + объект:', duplicate_keys)
display(strict_df['elementary_obj_type'].fillna('empty').value_counts(dropna=False))

Повторных ключей задача + объект: 0


elementary_obj_type
nedv_ul_and_ip    1211
Name: count, dtype: int64

# 5. Получение ФИАС из CDI

Для поиска используется доступная таблица `DM_RISK_AVATAR.STG_ADDRESS_CDI_ZUD`.

Связь строится в два шага:

```text
policyholder_cdi_id из Сферы
→ CLIENT_ID в CDI
→ адреса с признаком страхового объекта
→ сравнение с full_address объекта Сферы
```

`cdi_id` не считается связью с объектом. Он только ограничивает список адресами нужного страхователя. Связь принимается только после сравнения адреса.

Если после сравнения осталось несколько разных ФИАС, ничего автоматически не выбирается.

In [12]:
# готовим объекты для поиска адресов CDI
sphere_with_row_id = strict_df.copy()
sphere_with_row_id.insert(
    0,
    'sphere_row_id',
    range(1, len(sphere_with_row_id) + 1),
)

sphere_with_row_id['source_address'] = (
    sphere_with_row_id['full_address']
    .astype('string')
    .str.strip()
    .replace('', pd.NA)
)

cdi_input_columns = [
    'sphere_row_id',
    'contract_id',
    'object_id',
    'policyholder_cdi_id',
    'policyholder_inn',
    'as_of_date',
    'source_address',
    'total_area',
]

missing_cdi_input = [
    column for column in cdi_input_columns
    if column not in sphere_with_row_id.columns
]
if missing_cdi_input:
    raise KeyError(
        'Для CDI не хватает колонок: '
        + ', '.join(missing_cdi_input)
    )

def json_scalar(value):
    if value is None or pd.isna(value):
        return None
    return str(value)

cdi_records = []
for row in sphere_with_row_id[cdi_input_columns].itertuples(
    index=False,
    name=None,
):
    cdi_records.append({
        column: (
            int(value)
            if column == 'sphere_row_id'
            else json_scalar(value)
        )
        for column, value in zip(cdi_input_columns, row)
    })

cdi_json = json.dumps(cdi_records, ensure_ascii=False)

print('Строк объектов:', len(sphere_with_row_id))
print(
    'Строк с CDI ID:',
    sphere_with_row_id['policyholder_cdi_id'].notna().sum(),
)
print(
    'Строк с full_address:',
    sphere_with_row_id['source_address'].notna().sum(),
)

Строк объектов: 1211
Строк с CDI ID: 0
Строк с full_address: 1057


In [13]:
cdi_table_sql = r"""
/*
Ищем адреса страховых объектов в CDI.
Запрос только читает таблицы КХД.
*/

with sphere_objects as (
    select /*+ materialize */
        s.sphere_row_id,
        trim(s.cdi_id) as cdi_id,
        regexp_replace(s.inn, '[^0-9]', '') as inn,
        case
            when regexp_like(s.as_of_date, '^[0-9]{4}-[0-9]{2}-[0-9]{2}')
            then to_date(substr(s.as_of_date, 1, 10), 'YYYY-MM-DD')
        end as object_date
    from json_table(
        :cdi_json,
        '$[*]'
        columns (
            sphere_row_id number path '$.sphere_row_id',
            cdi_id varchar2(500) path '$.policyholder_cdi_id',
            inn varchar2(50) path '$.policyholder_inn',
            as_of_date varchar2(100) path '$.as_of_date'
        )
    ) s
),

resolved_clients_raw as (
    select distinct
        s.sphere_row_id,
        trim(s.cdi_id) as khd_client_id,
        'direct_client_id' as client_match_method,
        1 as client_match_priority
    from sphere_objects s
    where s.cdi_id is not null
      and exists (
          select 1
          from DM_RISK_AVATAR.STG_ADDRESS_CDI_ZUD a
          where trim(a.client_id) = s.cdi_id
      )

    union all

    select distinct
        s.sphere_row_id,
        trim(p.party_hidparty) as khd_client_id,
        'party_hidparty' as client_match_method,
        2 as client_match_priority
    from sphere_objects s
    join DM_RISK_AVATAR.STG_PARTY_SRC p
        on trim(p.party_hidparty) = s.cdi_id
    where s.cdi_id is not null

    union all

    select distinct
        s.sphere_row_id,
        trim(p.party_hidparty) as khd_client_id,
        'party_source_id' as client_match_method,
        3 as client_match_priority
    from sphere_objects s
    join DM_RISK_AVATAR.STG_PARTY_SRC p
        on trim(p.source_id) = s.cdi_id
    where s.cdi_id is not null
      and p.party_hidparty is not null
),

resolved_clients as (
    select
        r.sphere_row_id,
        r.khd_client_id,
        min(r.client_match_method) keep (
            dense_rank first order by r.client_match_priority
        ) as client_match_method,
        min(r.client_match_priority) as client_match_priority
    from resolved_clients_raw r
    group by r.sphere_row_id, r.khd_client_id
),

address_candidates_raw as (
    select
        r.sphere_row_id,
        r.khd_client_id,
        r.client_match_method,
        r.client_match_priority,
        a.address_id,
        a.address_name,
        a.fias_id_house,
        a.fias_id,
        a.fias_level,
        a.city,
        a.settlement,
        a.street,
        a.house_number,
        a.korpus,
        a.stroenie,
        a.vladenie,
        a.flat,
        a.flat2,
        a.office,
        a.office2,
        a.room,
        a.room2,
        a.verified_address_flag,
        a.insured_start_date,
        a.insured_end_date
    from resolved_clients r
    join sphere_objects s
        on s.sphere_row_id = r.sphere_row_id
    join DM_RISK_AVATAR.STG_ADDRESS_CDI_ZUD a
        on trim(a.client_id) = r.khd_client_id
    where a.insured_object_address_flag = 1
      and (
          s.object_date is null
          or (
              (a.insured_start_date is null
               or a.insured_start_date <= s.object_date)
              and (a.insured_end_date is null
                   or a.insured_end_date >= s.object_date)
          )
      )
),

address_candidates as (
    select a.*
    from (
        select
            a.*,
            row_number() over (
                partition by a.sphere_row_id, a.address_id
                order by
                    a.client_match_priority,
                    a.khd_client_id
            ) as address_row_number
        from address_candidates_raw a
    ) a
    where a.address_row_number = 1
)

select /*+ no_parallel */
    s.sphere_row_id,
    r.khd_client_id as cdi_client_id,
    r.client_match_method as cdi_client_match_method,
    a.address_id as cdi_address_id,
    a.address_name as cdi_address_name,
    a.fias_id_house as cdi_fias_id_house_raw,
    a.fias_id as cdi_fias_id_raw,
    a.fias_level as cdi_fias_level,
    a.city as cdi_city,
    a.settlement as cdi_settlement,
    a.street as cdi_street,
    a.house_number as cdi_house_number,
    a.korpus as cdi_korpus,
    a.stroenie as cdi_stroenie,
    a.vladenie as cdi_vladenie,
    a.flat as cdi_flat,
    a.flat2 as cdi_flat2,
    a.office as cdi_office,
    a.office2 as cdi_office2,
    a.room as cdi_room,
    a.room2 as cdi_room2,
    a.verified_address_flag as cdi_verified_address_flag
from sphere_objects s
left join resolved_clients r
    on r.sphere_row_id = s.sphere_row_id
left join address_candidates a
    on a.sphere_row_id = s.sphere_row_id
order by s.sphere_row_id, a.address_id
"""

if not KHD_DATA_SCHEMA.replace('_', '').isalnum():
    raise ValueError('Некорректное имя схемы КХД')

cdi_query = cdi_table_sql.replace(
    'DM_RISK_AVATAR.',
    f'{KHD_DATA_SCHEMA.upper()}.',
)

with khd_connection.cursor() as cursor:
    cdi_json_bind = cursor.var(oracledb.DB_TYPE_CLOB)
    cdi_json_bind.setvalue(0, cdi_json)
    cursor.execute(cdi_query, cdi_json=cdi_json_bind)
    cdi_columns = [
        str(column[0]).lower()
        for column in cursor.description
    ]
    cdi_rows = cursor.fetchall()

cdi_lookup_df = pd.DataFrame(cdi_rows, columns=cdi_columns)

if cdi_lookup_df.empty:
    raise RuntimeError('CDI не вернул ни одной строки')

print('Строк кандидатов CDI:', len(cdi_lookup_df))
print(
    'Объектов с найденным CDI-клиентом:',
    cdi_lookup_df.loc[
        cdi_lookup_df['cdi_client_id'].notna(),
        'sphere_row_id',
    ].nunique(),
)
print(
    'Объектов с адресами CDI:',
    cdi_lookup_df.loc[
        cdi_lookup_df['cdi_address_id'].notna(),
        'sphere_row_id',
    ].nunique(),
)

Строк кандидатов CDI: 1211
Объектов с найденным CDI-клиентом: 0
Объектов с адресами CDI: 0


In [14]:
# сравниваем адрес Сферы с адресами CDI
def normalize_address_text(value):
    if value is None or pd.isna(value):
        return ''
    result = str(value).lower().replace('ё', 'е')
    result = re.sub(
        r'\b(российская федерация|россия|город|г|'
        r'улица|ул|дом|д|корпус|корп|к|'
        r'строение|стр|ст|квартира|кв|'
        r'офис|помещение|пом|комната|комн)\b',
        ' ',
        result,
    )
    return re.sub(r'[^0-9a-zа-я]+', '', result)


def normalize_part(value):
    if value is None or pd.isna(value):
        return ''
    result = str(value).lower().replace('ё', 'е')
    return re.sub(r'[^0-9a-zа-я]+', '', result)


candidate_rows = cdi_lookup_df.loc[
    cdi_lookup_df['cdi_address_id'].notna()
].copy()

source_columns = [
    'sphere_row_id',
    'source_address',
    'settlement',
    'street',
    'house',
    'building',
    'block',
    'flat',
    'office',
]
candidate_rows = candidate_rows.merge(
    sphere_with_row_id[source_columns],
    on='sphere_row_id',
    how='left',
    validate='many_to_one',
)

candidate_rows['source_address_key'] = (
    candidate_rows['source_address'].map(normalize_address_text)
)
candidate_rows['cdi_address_key'] = (
    candidate_rows['cdi_address_name'].map(normalize_address_text)
)
candidate_rows['full_address_matches'] = (
    candidate_rows['source_address_key'].ne('')
    & candidate_rows['source_address_key'].eq(
        candidate_rows['cdi_address_key']
    )
)

for column in [
    'settlement', 'street', 'house', 'building', 'block', 'flat', 'office',
    'cdi_city', 'cdi_settlement', 'cdi_street', 'cdi_house_number',
    'cdi_vladenie', 'cdi_korpus', 'cdi_stroenie', 'cdi_flat',
    'cdi_flat2', 'cdi_office', 'cdi_office2', 'cdi_room', 'cdi_room2',
]:
    candidate_rows[column + '_key'] = (
        candidate_rows[column].map(normalize_part)
    )

candidate_rows['sphere_locality_key'] = candidate_rows['settlement_key']
candidate_rows['cdi_locality_key'] = candidate_rows['cdi_city_key'].where(
    candidate_rows['cdi_city_key'].ne(''),
    candidate_rows['cdi_settlement_key'],
)
candidate_rows['cdi_house_key'] = (
    candidate_rows['cdi_house_number_key'].where(
        candidate_rows['cdi_house_number_key'].ne(''),
        candidate_rows['cdi_vladenie_key'],
    )
)

locality_matches = (
    candidate_rows['sphere_locality_key'].eq('')
    | candidate_rows['cdi_locality_key'].eq('')
    | candidate_rows['sphere_locality_key'].eq(
        candidate_rows['cdi_locality_key']
    )
)
street_matches = (
    candidate_rows['street_key'].ne('')
    & candidate_rows['street_key'].eq(candidate_rows['cdi_street_key'])
)
house_matches = (
    candidate_rows['house_key'].ne('')
    & candidate_rows['house_key'].eq(candidate_rows['cdi_house_key'])
)

building_matches = (
    candidate_rows['building_key'].eq('')
    | candidate_rows['building_key'].eq(
        candidate_rows['cdi_stroenie_key']
    )
)
block_matches = (
    candidate_rows['block_key'].eq('')
    | candidate_rows['block_key'].eq(candidate_rows['cdi_korpus_key'])
)

cdi_premise_values = candidate_rows[
    [
        'cdi_flat_key', 'cdi_flat2_key',
        'cdi_office_key', 'cdi_office2_key',
        'cdi_room_key', 'cdi_room2_key',
    ]
].apply(lambda row: {value for value in row if value}, axis=1)
source_premise = candidate_rows['flat_key'].where(
    candidate_rows['flat_key'].ne(''),
    candidate_rows['office_key'],
)
premise_matches = pd.Series(
    [
        not source_value or source_value in cdi_values
        for source_value, cdi_values in zip(
            source_premise,
            cdi_premise_values,
        )
    ],
    index=candidate_rows.index,
)

candidate_rows['structured_address_matches'] = (
    locality_matches
    & street_matches
    & house_matches
    & building_matches
    & block_matches
    & premise_matches
)
candidate_rows['address_matches'] = (
    candidate_rows['full_address_matches']
    | candidate_rows['structured_address_matches']
)

candidate_rows['cdi_match_key'] = (
    candidate_rows['cdi_fias_id_raw']
    .astype('string')
    .str.strip()
    .replace('', pd.NA)
    .fillna(
        candidate_rows['cdi_fias_id_house_raw']
        .astype('string')
        .str.strip()
        .replace('', pd.NA)
    )
    .fillna(candidate_rows['cdi_address_key'].replace('', pd.NA))
)

candidate_count = (
    candidate_rows.groupby('sphere_row_id')['cdi_address_id']
    .nunique()
    .rename('cdi_address_candidate_count')
)

matched_rows = (
    candidate_rows.loc[candidate_rows['address_matches']]
    .drop_duplicates(['sphere_row_id', 'cdi_address_id'])
)
matched_key_count = (
    matched_rows.groupby('sphere_row_id')['cdi_match_key']
    .nunique()
    .rename('cdi_address_match_key_count')
)
matched_address_count = (
    matched_rows.groupby('sphere_row_id')['cdi_address_id']
    .nunique()
    .rename('cdi_address_match_count')
)

unique_match_ids = set(
    matched_key_count.loc[matched_key_count.eq(1)].index
)
chosen_rows = (
    matched_rows.loc[
        matched_rows['sphere_row_id'].isin(unique_match_ids)
    ]
    .sort_values(
        ['sphere_row_id', 'full_address_matches', 'cdi_address_id'],
        ascending=[True, False, True],
    )
    .drop_duplicates('sphere_row_id')
)

chosen_columns = [
    'sphere_row_id',
    'cdi_client_id',
    'cdi_client_match_method',
    'cdi_address_id',
    'cdi_address_name',
    'cdi_fias_id_house_raw',
    'cdi_fias_id_raw',
    'cdi_fias_level',
    'full_address_matches',
    'structured_address_matches',
]
chosen_rows = chosen_rows[chosen_columns]

cdi_address_df = sphere_with_row_id.merge(
    chosen_rows,
    on='sphere_row_id',
    how='left',
    validate='one_to_one',
)
cdi_address_df = cdi_address_df.merge(
    candidate_count,
    on='sphere_row_id',
    how='left',
    validate='one_to_one',
)
cdi_address_df = cdi_address_df.merge(
    matched_address_count,
    on='sphere_row_id',
    how='left',
    validate='one_to_one',
)
cdi_address_df = cdi_address_df.merge(
    matched_key_count,
    on='sphere_row_id',
    how='left',
    validate='one_to_one',
)

for column in [
    'cdi_address_candidate_count',
    'cdi_address_match_count',
    'cdi_address_match_key_count',
]:
    cdi_address_df[column] = (
        cdi_address_df[column].fillna(0).astype('int64')
    )

cdi_address_df['cdi_address_match_status'] = 'address_not_matched'
cdi_address_df.loc[
    cdi_address_df['policyholder_cdi_id'].isna(),
    'cdi_address_match_status',
] = 'no_cdi_id'
cdi_address_df.loc[
    cdi_address_df['source_address'].isna(),
    'cdi_address_match_status',
] = 'no_source_address'
cdi_address_df.loc[
    cdi_address_df['cdi_address_candidate_count'].eq(0)
    & cdi_address_df['policyholder_cdi_id'].notna()
    & cdi_address_df['source_address'].notna(),
    'cdi_address_match_status',
] = 'no_cdi_object_addresses'
cdi_address_df.loc[
    cdi_address_df['cdi_address_match_key_count'].gt(1),
    'cdi_address_match_status',
] = 'ambiguous_address_match'
cdi_address_df.loc[
    cdi_address_df['cdi_address_match_key_count'].eq(1),
    'cdi_address_match_status',
] = 'unique_cdi_address'

cdi_address_df['cdi_fias_id_house'] = (
    cdi_address_df['cdi_fias_id_house_raw']
    .astype('string')
    .str.strip()
    .replace('', pd.NA)
)
cdi_address_df['cdi_fias_id_flat'] = pd.NA
is_flat_level = (
    cdi_address_df['cdi_fias_level']
    .astype('string')
    .str.upper()
    .str.strip()
    .isin(['FIAS_FLAT', 'FLAT'])
)
cdi_address_df.loc[
    is_flat_level,
    'cdi_fias_id_flat',
] = (
    cdi_address_df.loc[is_flat_level, 'cdi_fias_id_raw']
    .astype('string')
    .str.strip()
    .replace('', pd.NA)
)

cdi_address_df['cdi_match_status'] = 'not_found'
cdi_address_df.loc[
    cdi_address_df['cdi_address_match_status'].eq('no_source_address'),
    'cdi_match_status',
] = 'no_source_address'
cdi_address_df.loc[
    cdi_address_df['cdi_address_match_status'].eq('ambiguous_address_match'),
    'cdi_match_status',
] = 'ambiguous_house_fias'
cdi_address_df.loc[
    cdi_address_df['cdi_address_match_status'].eq('unique_cdi_address')
    & cdi_address_df['cdi_fias_id_house'].notna(),
    'cdi_match_status',
] = 'unique_house_fias'
cdi_address_df['cdi_is_unique_match'] = (
    cdi_address_df['cdi_match_status']
    .eq('unique_house_fias')
    .astype('int64')
)

cdi_address_df['cdi_flat_match_status'] = 'flat_fias_not_found'
cdi_address_df.loc[
    cdi_address_df['cdi_address_match_status'].eq('unique_cdi_address')
    & cdi_address_df['cdi_fias_id_flat'].notna(),
    'cdi_flat_match_status',
] = 'unique_flat_fias'
cdi_address_df.loc[
    cdi_address_df['cdi_address_match_status'].eq('ambiguous_address_match'),
    'cdi_flat_match_status',
] = 'ambiguous_flat_fias'
cdi_address_df['cdi_flat_is_unique_match'] = (
    cdi_address_df['cdi_flat_match_status']
    .eq('unique_flat_fias')
    .astype('int64')
)

cdi_address_df['cdi_house_fias_candidate_count'] = (
    cdi_address_df['cdi_is_unique_match']
)
cdi_address_df['cdi_flat_fias_candidate_count'] = (
    cdi_address_df['cdi_flat_is_unique_match']
)

flat_value = cdi_address_df['flat'].astype('string').str.strip()
office_value = cdi_address_df['office'].astype('string').str.strip()
cdi_address_df['sphere_has_premise_in_address'] = (
    (flat_value.notna() & flat_value.ne(''))
    | (office_value.notna() & office_value.ne(''))
)

print(cdi_address_df['cdi_address_match_status'].value_counts(dropna=False))
print()
print('Статусы ФИАС дома:')
print(cdi_address_df['cdi_match_status'].value_counts(dropna=False))
print()
print('Статусы ФИАС помещения:')
print(cdi_address_df['cdi_flat_match_status'].value_counts(dropna=False))

cdi_address_match_status
no_cdi_id            1057
no_source_address     154
Name: count, dtype: int64

Статусы ФИАС дома:
cdi_match_status
not_found            1057
no_source_address     154
Name: count, dtype: int64

Статусы ФИАС помещения:
cdi_flat_match_status
flat_fias_not_found    1211
Name: count, dtype: int64


# 6. Поиск здания в ЕГРН

В ЕГРН передаётся только однозначный ФИАС дома, полученный из текста адреса через CDI.

Если по ФИАС найдено одно здание, оно присоединяется. Если зданий несколько, площадь используется как дополнительная проверка. При отсутствии площади несколько кандидатов не выбираются.


In [15]:
# готовим ФИАС дома и площадь для поиска ЕГРН
egrn_input = cdi_address_df.loc[
    cdi_address_df['cdi_is_unique_match'].eq(1),
    ['sphere_row_id', 'cdi_fias_id_house', 'total_area'],
].copy()

def json_scalar(value):
    if value is None or pd.isna(value):
        return None
    return str(value)

egrn_records = [
    {
        'sphere_row_id': int(row.sphere_row_id),
        'fias_id_house': json_scalar(row.cdi_fias_id_house),
        'total_area': json_scalar(row.total_area),
    }
    for row in egrn_input.itertuples(index=False)
]
egrn_json = json.dumps(egrn_records, ensure_ascii=False)

print('Строк передано в поиск ЕГРН:', len(egrn_records))


Строк передано в поиск ЕГРН: 0


In [16]:
egrn_by_fias_sql = r"""
/*
Запрос ищет здания ЕГРН по ФИАС дома, полученному из CDI.

ФИАС дома используется только для формирования кандидатов. Если найдено
несколько кадастровых зданий, дополнительно проверяется площадь. Данные ЕГРН
возвращаются только при одном кандидате после проверки.
*/

with sphere_objects as (
    select /*+ materialize */
        s.sphere_row_id,
        trim(s.fias_id_house) as fias_id_house,
        replace(
            regexp_replace(trim(s.total_area), '[[:space:]]+', ''),
            ',',
            '.'
        ) as sphere_area_text
    from json_table(
        :egrn_json,
        '$[*]'
        columns (
            sphere_row_id number path '$.sphere_row_id',
            fias_id_house varchar2(500) path '$.fias_id_house',
            total_area varchar2(200) path '$.total_area'
        )
    ) s
),

sphere_prepared as (
    select
        s.*,
        case
            when regexp_like(s.sphere_area_text, '^[0-9]+([.][0-9]+)?$')
            then to_number(
                s.sphere_area_text,
                '999999999999999999999999D9999999999',
                'NLS_NUMERIC_CHARACTERS=''.,'''
            )
        end as sphere_area
    from sphere_objects s
),

egrn_raw as (
    select /*+ no_parallel(e) */
        s.sphere_row_id,
        s.sphere_area,
        coalesce(
            nullif(trim(e.cadaster), ''),
            'CAD_IND:' || cast(e.cad_ind as varchar2(200))
        ) as egrn_key,
        e.cad_ind,
        e.cadaster,
        e.egrn_address,
        e.square,
        case
            when regexp_like(
                replace(
                    regexp_replace(
                        trim(cast(e.square as varchar2(200))),
                        '[[:space:]]+',
                        ''
                    ),
                    ',',
                    '.'
                ),
                '^[0-9]+([.][0-9]+)?$'
            )
            then to_number(
                replace(
                    regexp_replace(
                        trim(cast(e.square as varchar2(200))),
                        '[[:space:]]+',
                        ''
                    ),
                    ',',
                    '.'
                ),
                '999999999999999999999999D9999999999',
                'NLS_NUMERIC_CHARACTERS=''.,'''
            )
        end as egrn_area,
        e.measure,
        e.building_type,
        e.oks_type,
        e.oks_purpose,
        e.object_status,
        e.fias_level,
        e.fias_id_house,
        e.row_update_date,
        e.ias_update_date
    from sphere_prepared s
    join DM_RISK_AVATAR.EGRN_DATA e
        on e.fias_id_house = s.fias_id_house
    where upper(trim(e.fias_level)) = 'FIAS_HOUSE'
      and lower(trim(e.oks_type)) in (
          'здание',
          'сооружение',
          'строение'
      )
      and e.flat is null
      and e.flat2 is null
      and e.office is null
      and e.office2 is null
      and e.room is null
      and e.room2 is null
      and e.compartment1 is null
      and e.compartment2 is null
      and (e.cadaster is not null or e.cad_ind is not null)
),

ranked_egrn as (
    select
        e.*,
        row_number() over (
            partition by e.sphere_row_id, e.egrn_key
            order by
                e.row_update_date desc nulls last,
                e.ias_update_date desc nulls last,
                e.cad_ind desc nulls last
        ) as version_number
    from egrn_raw e
),

one_row_per_object as (
    select e.*
    from ranked_egrn e
    where e.version_number = 1
),

area_check as (
    select
        e.*,
        count(*) over (
            partition by e.sphere_row_id
        ) as address_candidate_count,
        case
            when e.sphere_area > 0
             and e.egrn_area is not null
             and abs(e.egrn_area - e.sphere_area)
                 <= greatest(1, e.sphere_area * 0.01)
                then 1
            else 0
        end as area_matches
    from one_row_per_object e
),

area_choice as (
    select
        e.*,
        max(e.area_matches) over (
            partition by e.sphere_row_id
        ) as has_area_match
    from area_check e
),

candidates_after_area as (
    select e.*
    from area_choice e
    where e.has_area_match = 0
       or e.area_matches = 1
),

candidate_counts as (
    select
        e.*,
        count(*) over (
            partition by e.sphere_row_id
        ) as candidate_count
    from candidates_after_area e
),

candidate_summary as (
    select
        e.sphere_row_id,
        max(e.address_candidate_count) as address_candidate_count,
        max(e.candidate_count) as candidate_count,
        max(e.has_area_match) as has_area_match
    from candidate_counts e
    group by e.sphere_row_id
),

chosen_egrn as (
    select e.*
    from candidate_counts e
    where e.candidate_count = 1
)

select /*+ no_parallel */
    s.sphere_row_id as "sphere_row_id",
    s.fias_id_house as "cdi_fias_id_house",
    nvl(cs.address_candidate_count, 0) as "egrn_address_candidate_count",
    nvl(cs.candidate_count, 0) as "egrn_candidate_count",
    case
        when cs.candidate_count = 1 then 1
        else 0
    end as "egrn_is_unique_match",
    case
        when nvl(cs.candidate_count, 0) = 0
            then 'not_found'
        when cs.candidate_count > 1
            then 'ambiguous'
        when cs.address_candidate_count > 1
         and cs.has_area_match = 1
            then 'fias_house_and_area'
        else 'fias_house_only'
    end as "egrn_match_method",
    e.cad_ind as "egrn_cad_ind",
    e.cadaster as "egrn_cadaster",
    e.egrn_address as "egrn_address",
    e.square as "egrn_square",
    e.measure as "egrn_measure",
    e.building_type as "egrn_building_type",
    e.oks_type as "egrn_oks_type",
    e.oks_purpose as "egrn_oks_purpose",
    e.object_status as "egrn_object_status",
    e.fias_level as "egrn_fias_level",
    e.fias_id_house as "egrn_fias_id_house"
from sphere_prepared s
left join candidate_summary cs
    on cs.sphere_row_id = s.sphere_row_id
left join chosen_egrn e
    on e.sphere_row_id = s.sphere_row_id
order by s.sphere_row_id

"""


In [17]:
egrn_expected_columns = [
    'sphere_row_id',
    'cdi_fias_id_house',
    'egrn_address_candidate_count',
    'egrn_candidate_count',
    'egrn_is_unique_match',
    'egrn_match_method',
    'egrn_cad_ind',
    'egrn_cadaster',
    'egrn_address',
    'egrn_square',
    'egrn_measure',
    'egrn_building_type',
    'egrn_oks_type',
    'egrn_oks_purpose',
    'egrn_object_status',
    'egrn_fias_level',
    'egrn_fias_id_house',
]

khd_schema = KHD_DATA_SCHEMA.upper()
if not khd_schema.replace('_', '').isalnum():
    raise ValueError('Некорректное имя схемы КХД')

if egrn_records:
    egrn_query = egrn_by_fias_sql.replace(
        'DM_RISK_AVATAR.',
        f'{khd_schema}.',
    )
    with khd_connection.cursor() as cursor:
        egrn_json_bind = cursor.var(oracledb.DB_TYPE_CLOB)
        egrn_json_bind.setvalue(0, egrn_json)
        cursor.execute(egrn_query, egrn_json=egrn_json_bind)
        egrn_columns = [
            str(column[0]).lower()
            for column in cursor.description
        ]
        egrn_rows = cursor.fetchall()
    egrn_lookup_df = pd.DataFrame(egrn_rows, columns=egrn_columns)
else:
    egrn_lookup_df = pd.DataFrame(columns=egrn_expected_columns)

missing_egrn_columns = sorted(
    set(egrn_expected_columns) - set(egrn_lookup_df.columns)
)
if missing_egrn_columns:
    raise ValueError(
        'ЕГРН не вернул ожидаемые колонки: '
        + ', '.join(missing_egrn_columns)
    )
if egrn_lookup_df['sphere_row_id'].duplicated().any():
    raise ValueError('ЕГРН вернул несколько итоговых строк для объекта Сферы')

strict_address_egrn_df = cdi_address_df.merge(
    egrn_lookup_df[egrn_expected_columns],
    on='sphere_row_id',
    how='left',
    validate='one_to_one',
    suffixes=('', '_egrn_result'),
)
strict_address_egrn_df['egrn_is_unique_match'] = (
    strict_address_egrn_df['egrn_is_unique_match']
    .fillna(0)
    .astype('int64')
)

strict_address_egrn_df['pipeline_match_status'] = 'not_linked'
strict_address_egrn_df.loc[
    strict_address_egrn_df['cdi_is_unique_match'].eq(1),
    'pipeline_match_status',
] = 'cdi_address_linked'
strict_address_egrn_df.loc[
    strict_address_egrn_df['egrn_is_unique_match'].eq(1),
    'pipeline_match_status',
] = 'egrn_linked'

if len(strict_address_egrn_df) != len(strict_df):
    raise ValueError('После CDI и ЕГРН изменилось количество строк датасета')

print('Строк в итоговом датасете:', len(strict_address_egrn_df))


Строк в итоговом датасете: 1211


# 7. Проверка результата

В проверке показано, сколько адресов CDI и объектов ЕГРН удалось присоединить однозначно. Реальные адреса и номера договоров здесь не выводятся.


In [18]:
quality_profile = pd.DataFrame({
    'Показатель': [
        'Строк исходного строгого датасета',
        'Строк итогового датасета',
        'Строк без адреса Сферы',
        'Однозначных ФИАС дома из CDI',
        'Неоднозначных ФИАС дома из CDI',
        'Строк с явным помещением в адресе',
        'Однозначных ФИАС помещения из CDI',
        'Явное помещение и один ФИАС помещения',
        'Однозначных зданий ЕГРН',
        'Неоднозначных зданий ЕГРН',
    ],
    'Значение': [
        len(strict_df),
        len(strict_address_egrn_df),
        strict_address_egrn_df['cdi_match_status']
            .eq('no_source_address').sum(),
        strict_address_egrn_df['cdi_is_unique_match'].sum(),
        strict_address_egrn_df['cdi_match_status']
            .eq('ambiguous_house_fias').sum(),
        strict_address_egrn_df['sphere_has_premise_in_address'].sum(),
        strict_address_egrn_df['cdi_flat_is_unique_match'].sum(),
        (
            strict_address_egrn_df['sphere_has_premise_in_address']
            & strict_address_egrn_df['cdi_flat_is_unique_match'].eq(1)
        ).sum(),
        strict_address_egrn_df['egrn_is_unique_match'].sum(),
        strict_address_egrn_df['egrn_match_method']
            .eq('ambiguous').sum(),
    ],
})
display(quality_profile)

print()
print('Статусы CDI по дому:')
print(strict_address_egrn_df['cdi_match_status'].value_counts(dropna=False))
print()
print('Статусы CDI по помещению:')
print(strict_address_egrn_df['cdi_flat_match_status'].value_counts(dropna=False))
print()
print('Методы ЕГРН по зданию:')
print(strict_address_egrn_df['egrn_match_method'].value_counts(dropna=False))


,Показатель,Значение
0,Строк исходного строгого датасета,1211
1,Строк итогового датасета,1211
2,Строк без адреса Сферы,154
3,Однозначных ФИАС дома из CDI,0
4,Неоднозначных ФИАС дома из CDI,0
5,Строк с явным помещением в адресе,0
6,Однозначных ФИАС помещения из CDI,0
7,Явное помещение и один ФИАС помещения,0
8,Однозначных зданий ЕГРН,0
9,Неоднозначных зданий ЕГРН,0



Статусы CDI по дому:
cdi_match_status
not_found            1057
no_source_address     154
Name: count, dtype: int64

Статусы CDI по помещению:
cdi_flat_match_status
flat_fias_not_found    1211
Name: count, dtype: int64

Методы ЕГРН по зданию:
egrn_match_method
NaN    1211
Name: count, dtype: int64


# 8. Где теряются объекты

Этот раздел нужен, чтобы не смотреть только на общий процент соединения.

Проверяются переходы:

```text
объект Сферы
→ заполнен full_address
→ CDI вернул один ФИАС дома
→ в ЕГРН нашлись здания с этим ФИАС
→ остался один кандидат
```

Площадь используется только тогда, когда по ФИАС дома нашлось несколько зданий.

In [19]:
diagnostic_df = strict_address_egrn_df.copy()

diagnostic_df['area_for_match'] = pd.to_numeric(
    diagnostic_df['total_area'],
    errors='coerce',
)
diagnostic_df['has_area_for_match'] = (
    diagnostic_df['area_for_match'].gt(0)
)

for column in [
    'egrn_address_candidate_count',
    'egrn_candidate_count',
]:
    diagnostic_df[column] = pd.to_numeric(
        diagnostic_df[column],
        errors='coerce',
    ).fillna(0).astype('int64')

# причина записывается по тому этапу, на котором остановилась строка
diagnostic_df['diagnostic_reason'] = 'другая причина'

diagnostic_df.loc[
    diagnostic_df['cdi_match_status'].eq('no_source_address'),
    'diagnostic_reason',
] = 'нет full_address в Сфере'

diagnostic_df.loc[
    diagnostic_df['cdi_match_status'].eq('lookup_error'),
    'diagnostic_reason',
] = 'ошибка вызова CDI'

diagnostic_df.loc[
    diagnostic_df['cdi_match_status'].eq('not_found'),
    'diagnostic_reason',
] = 'CDI не вернул ФИАС дома'

diagnostic_df.loc[
    diagnostic_df['cdi_match_status'].eq('ambiguous_house_fias'),
    'diagnostic_reason',
] = 'CDI вернул несколько ФИАС дома'

unique_cdi = diagnostic_df['cdi_is_unique_match'].eq(1)
no_egrn_candidates = diagnostic_df['egrn_address_candidate_count'].eq(0)
ambiguous_egrn = (
    diagnostic_df['egrn_address_candidate_count'].gt(0)
    & diagnostic_df['egrn_is_unique_match'].eq(0)
)

diagnostic_df.loc[
    unique_cdi & no_egrn_candidates,
    'diagnostic_reason',
] = 'ФИАС дома есть, но здание ЕГРН не найдено'

diagnostic_df.loc[
    unique_cdi & ambiguous_egrn & ~diagnostic_df['has_area_for_match'],
    'diagnostic_reason',
] = 'несколько зданий ЕГРН, а площади нет'

area_narrowed = (
    diagnostic_df['egrn_candidate_count']
    < diagnostic_df['egrn_address_candidate_count']
)
diagnostic_df.loc[
    unique_cdi
    & ambiguous_egrn
    & diagnostic_df['has_area_for_match']
    & ~area_narrowed,
    'diagnostic_reason',
] = 'несколько зданий ЕГРН, площадь не помогла'

diagnostic_df.loc[
    unique_cdi
    & ambiguous_egrn
    & diagnostic_df['has_area_for_match']
    & area_narrowed,
    'diagnostic_reason',
] = 'площадь сузила поиск, но кандидатов осталось несколько'

diagnostic_df.loc[
    diagnostic_df['egrn_is_unique_match'].eq(1),
    'diagnostic_reason',
] = 'ЕГРН присоединён однозначно'

reason_summary = (
    diagnostic_df.groupby('diagnostic_reason', dropna=False)
    .agg(
        rows=('sphere_row_id', 'size'),
        unique_objects=('object_id', 'nunique'),
        unique_addresses=('source_address', 'nunique'),
        rows_with_area=('has_area_for_match', 'sum'),
    )
    .reset_index()
    .sort_values('rows', ascending=False)
)
reason_summary['pct_of_all_rows'] = (
    reason_summary['rows'] / len(diagnostic_df) * 100
).round(2)

display(reason_summary)

,diagnostic_reason,rows,unique_objects,unique_addresses,rows_with_area,pct_of_all_rows
0,CDI не вернул ФИАС дома,1057,1049,810,360,87.28
1,нет full_address в Сфере,154,152,0,52,12.72


In [20]:
# воронка показывает потерю между соседними этапами
funnel_steps = [
    ('все объекты строгого датасета', pd.Series(True, index=diagnostic_df.index)),
    ('есть full_address', diagnostic_df['source_address'].notna()),
    ('CDI вернул один ФИАС дома', diagnostic_df['cdi_is_unique_match'].eq(1)),
    ('в ЕГРН есть хотя бы один кандидат', diagnostic_df['egrn_address_candidate_count'].gt(0)),
    ('ЕГРН присоединён однозначно', diagnostic_df['egrn_is_unique_match'].eq(1)),
]

funnel_rows = []
previous_count = None
total_count = len(diagnostic_df)

for step_number, (step_name, step_mask) in enumerate(funnel_steps, start=1):
    row_count = int(step_mask.sum())
    funnel_rows.append({
        'step_number': step_number,
        'step': step_name,
        'rows': row_count,
        'pct_of_all_rows': round(row_count / total_count * 100, 2),
        'pct_of_previous_step': (
            100.0
            if previous_count is None
            else round(row_count / previous_count * 100, 2)
            if previous_count
            else 0.0
        ),
    })
    previous_count = row_count

funnel_df = pd.DataFrame(funnel_rows)
display(funnel_df)

,step_number,step,rows,pct_of_all_rows,pct_of_previous_step
0,1,все объекты строгого датасета,1211,100.00,100.00
1,2,есть full_address,1057,87.28,87.28
2,3,CDI вернул один ФИАС дома,0,0.00,0.00
3,4,в ЕГРН есть хотя бы один кандидат,0,0.00,0.00
4,5,ЕГРН присоединён однозначно,0,0.00,0.00


In [21]:
# отдельно смотрим, сколько кандидатов даёт один ФИАС дома
candidates_df = diagnostic_df.loc[
    diagnostic_df['cdi_is_unique_match'].eq(1)
].copy()

candidates_df['candidate_bucket'] = pd.cut(
    candidates_df['egrn_address_candidate_count'],
    bins=[-1, 0, 1, 5, 20, float('inf')],
    labels=['0', '1', '2-5', '6-20', 'больше 20'],
)

candidate_summary = (
    candidates_df.groupby(
        ['candidate_bucket', 'has_area_for_match'],
        observed=True,
        dropna=False,
    )
    .size()
    .rename('rows')
    .reset_index()
    .sort_values(['candidate_bucket', 'has_area_for_match'])
)

display(candidate_summary)

# проверяем, можно ли отдельно исследовать уровень квартиры или помещения
premise_profile = pd.DataFrame({
    'Показатель': [
        'явное помещение в full_address',
        'CDI вернул один ФИАС помещения',
        'явное помещение и один ФИАС помещения',
        'CDI вернул несколько ФИАС помещения',
    ],
    'Значение': [
        diagnostic_df['sphere_has_premise_in_address'].sum(),
        diagnostic_df['cdi_flat_is_unique_match'].sum(),
        (
            diagnostic_df['sphere_has_premise_in_address']
            & diagnostic_df['cdi_flat_is_unique_match'].eq(1)
        ).sum(),
        diagnostic_df['cdi_flat_match_status']
            .eq('ambiguous_flat_fias').sum(),
    ],
})
display(premise_profile)


,candidate_bucket,has_area_for_match,rows


,Показатель,Значение
0,явное помещение в full_address,0
1,CDI вернул один ФИАС помещения,0
2,явное помещение и один ФИАС помещения,0
3,CDI вернул несколько ФИАС помещения,0


In [22]:
# сохраняем только агрегаты, без адресов, ИНН и номеров договоров
funnel_export = funnel_df.rename(columns={'step': 'label'}).copy()
funnel_export.insert(0, 'record_type', 'funnel')
funnel_export['unique_objects'] = pd.NA
funnel_export['unique_addresses'] = pd.NA
funnel_export['rows_with_area'] = pd.NA

reason_export = reason_summary.rename(
    columns={'diagnostic_reason': 'label'}
).copy()
reason_export.insert(0, 'record_type', 'reason')
reason_export['step_number'] = pd.NA
reason_export['pct_of_previous_step'] = pd.NA

diagnostic_export = pd.concat(
    [funnel_export, reason_export],
    ignore_index=True,
    sort=False,
)

diagnostic_columns = [
    'record_type',
    'step_number',
    'label',
    'rows',
    'pct_of_all_rows',
    'pct_of_previous_step',
    'unique_objects',
    'unique_addresses',
    'rows_with_area',
]
diagnostic_export = diagnostic_export[diagnostic_columns]

diagnostic_file = OUTPUT_DIR / 'диагностика_CDI_ЕГРН.csv'
diagnostic_export.to_csv(
    diagnostic_file,
    index=False,
    encoding='utf-8-sig',
)

print('Диагностика сохранена:', diagnostic_file)

Диагностика сохранена: t:\Блок актуарных расчетов\Управление актуарных расчетов\Общая\Светова\риск моделирование\анализ таблиц\sql python\РЕЗУЛЬТАТЫ_ЛОКАЛЬНО\диагностика_CDI_ЕГРН.csv


# 9. Сохранение результатов


In [23]:
snapshot_at = pd.Timestamp.now(tz='Europe/Moscow').isoformat()
strict_address_egrn_df['external_snapshot_at'] = snapshot_at

final_path = OUTPUT_DIR / 'датасет_строгий_адрес_CDI_ЕГРН.csv'
cdi_candidates_path = OUTPUT_DIR / 'снимок_CDI_по_адресу.csv'
egrn_path = OUTPUT_DIR / 'снимок_ЕГРН_по_адресу_CDI.csv'

strict_address_egrn_df.to_csv(
    final_path,
    index=False,
    sep=';',
    encoding='utf-8-sig',
)

# сохраняем кандидатов CDI и результат сравнения адресов
candidate_rows.assign(snapshot_at=snapshot_at).to_csv(
    cdi_candidates_path,
    index=False,
    sep=';',
    encoding='utf-8-sig',
)

egrn_lookup_df.assign(snapshot_at=snapshot_at).to_csv(
    egrn_path,
    index=False,
    sep=';',
    encoding='utf-8-sig',
)

print('Итоговый датасет:', final_path)
print('Кандидаты CDI:', cdi_candidates_path)
print('Ответы ЕГРН:', egrn_path)


Итоговый датасет: t:\Блок актуарных расчетов\Управление актуарных расчетов\Общая\Светова\риск моделирование\анализ таблиц\sql python\РЕЗУЛЬТАТЫ_ЛОКАЛЬНО\датасет_строгий_адрес_CDI_ЕГРН.csv
Кандидаты CDI: t:\Блок актуарных расчетов\Управление актуарных расчетов\Общая\Светова\риск моделирование\анализ таблиц\sql python\РЕЗУЛЬТАТЫ_ЛОКАЛЬНО\снимок_CDI_по_адресу.csv
Ответы ЕГРН: t:\Блок актуарных расчетов\Управление актуарных расчетов\Общая\Светова\риск моделирование\анализ таблиц\sql python\РЕЗУЛЬТАТЫ_ЛОКАЛЬНО\снимок_ЕГРН_по_адресу_CDI.csv


# 10. Список уникальных ИНН

Пустые значения и повторы исключаются.


In [24]:
inn_column = next(
    (
        column
        for column in ['policyholder_inn', 'inn']
        if column in strict_address_egrn_df.columns
    ),
    None,
)
if inn_column is None:
    raise ValueError('В итоговом датасете не найдена колонка ИНН')

inn_df = (
    strict_address_egrn_df[[inn_column]]
    .rename(columns={inn_column: 'inn'})
    .assign(inn=lambda frame: frame['inn'].astype('string').str.strip())
    .loc[lambda frame: frame['inn'].notna() & frame['inn'].ne('')]
    .drop_duplicates()
    .sort_values('inn')
    .reset_index(drop=True)
)

inn_path = OUTPUT_DIR / 'inn.csv'
inn_df.to_csv(
    inn_path,
    index=False,
    sep=';',
    encoding='utf-8-sig',
)

print('Уникальных ИНН:', len(inn_df))
print('Файл:', inn_path)


Уникальных ИНН: 476
Файл: t:\Блок актуарных расчетов\Управление актуарных расчетов\Общая\Светова\риск моделирование\анализ таблиц\sql python\РЕЗУЛЬТАТЫ_ЛОКАЛЬНО\inn.csv


In [25]:
engine.dispose()
khd_connection.close()
print('Подключения закрыты')


Подключения закрыты
